# Akcelerator STFT (Short-Time Fourier Transform) w czasie rzeczywistym na platformie FPGA Kria KV260
**Raport końcowy z projektu**

* **Autor:** Mateusz Szczęśniak
* **Środowisko:** Kria KV260, Vivado 2025.2, Python 3.11

---

## 1. Wstęp teoretyczny

### Krótkoterminowa Transformata Fouriera (STFT)
Klasyczna Transformata Fouriera (FT) pozwala na analizę widmową sygnału, jednak traci ona zupełnie informację o czasie wystąpienia poszczególnych częstotliwości. Jest to poważna wada w przypadku sygnałów niestacjonarnych (np. mowa, muzyka, sygnały biomedyczne), których struktura częstotliwościowa zmienia się w czasie.

Rozwiązaniem tego problemu jest **Krótkoterminowa Transformata Fouriera (STFT)**. Polega ona na podziale sygnału na krótkie segmenty (ramki), pomnożeniu ich przez funkcję okna w celu minimalizacji przecieku widma (spectral leakage) i obliczeniu klasycznej transformaty Fouriera (FFT) dla każdej z ramek.

Matematycznie STFT definiuje się jako:
$$STFT\{x(t)\}(\tau, f) = \int_{-\infty}^{\infty} x(t) w(t - \tau) e^{-j 2 \pi f t} dt$$

W dziedzinie dyskretnej, dla sygnału $x[n]$ i okna $w[n]$ o długości $N$, STFT zapisuje się jako:
$$X(m, k) = \sum_{n=0}^{N-1} x[n + m \cdot H] w[n] e^{-j \frac{2 \pi k n}{N}}$$
gdzie:
* $m$ - indeks ramki w czasie,
* $k$ - indeks prążka częstotliwościowego ($0 \le k < N$),
* $H$ - przesunięcie okna (hop size) określające nakładanie się ramek (overlap),
* $w[n]$ - funkcja okna.

### Zasada nieoznaczoności Heisenberga-Gabora
Wybór długości okna $N$ wiąże się z fundamentalnym kompromisem czasowo-częstotliwościowym, wynikającym z zasady nieoznaczoności:
$$\Delta t \cdot \Delta f \ge \frac{1}{4\pi}$$

* **Krótkie okno (szerokopasmowe STFT):** wysoka rozdzielczość czasowa (dokładnie wiemy, kiedy zaszło zdarzenie), ale słaba rozdzielczość częstotliwościowa (prążki zlewają się ze sobą).
* **Długie okno (wąskopasmowe STFT):** wysoka rozdzielczość częstotliwościowa (dokładnie rozróżniamy bliskie częstotliwości), ale słaba rozdzielczość czasowa (rozmycie czasowe).

### Porównanie: STFT vs Transformata Falkowa (Wavelet Transform)
* **STFT:** Stosuje **stałą szerokość okna** dla wszystkich częstotliwości. Rozdzielczość w czasie i częstotliwości jest jednolita na całej płaszczyźnie czas-częstotliwość. Jest optymalna dla sygnałów, gdzie zjawiska częstotliwościowe mają podobną dynamikę w czasie.
* **Transformata Falkowa (CWT/DWT):** Stosuje **zmienną szerokość okna** (analiza wielorozdzielcza). Dla wysokich częstotliwości okno falki jest krótkie (dobra rozdzielczość czasowa, słaba częstotliwościowa), a dla niskich częstotliwości okno jest długie (dobra rozdzielczość częstotliwościowa, słaba czasowa).
  * *Kiedy stosować STFT:* Analiza sygnałów harmonicznych, audio o stałej strukturze rytmicznej i tonalnej, prosta implementacja sprzętowa (FFT jest wysoce zoptymalizowane w strukturach FPGA).
  * *Kiedy stosować falki:* Detekcja szybkich stanów przejściowych (np. impulsy w EKG, pęknięcia w diagnostyce drganiowej maszyn), sygnały o bardzo zróżnicowanej skali czasowej.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal

# 1. Generowanie sygnału testowego - Chirp (zamieciony sinus)
fs = 1000  # Częstotliwość próbkowania (Hz)
t = np.arange(0, 2.0, 1/fs)
# Sygnał o częstotliwości rosnącej liniowo od 50 Hz do 400 Hz
x = signal.chirp(t, f0=50, f1=400, t1=2.0, method='linear')

# 2. Obliczenie STFT za pomocą SciPy
nperseg = 128
noverlap = 96
f, t_spec, Zxx = signal.stft(x, fs=fs, nperseg=nperseg, noverlap=noverlap, window='hann')

# 3. Rysowanie wykresu czasowego i spektrogramu
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

ax1.plot(t, x, color='royalblue')
ax1.set_title("Sygnał testowy (Chirp) w dziedzinie czasu")
ax1.set_xlabel("Czas [s]")
ax1.set_ylabel("Amplituda")
ax1.grid(True)

pcm = ax2.pcolormesh(t_spec, f, 20*np.log10(np.abs(Zxx) + 1e-9), shading='gouraud', cmap='inferno')
ax2.set_title("Spektrogram STFT (Okienkowanie Hann)")
ax2.set_ylabel("Częstotliwość [Hz]")
ax2.set_xlabel("Czas [s]")
fig.colorbar(pcm, ax=ax2, label='Amplituda [dB]')
ax2.grid(True)

plt.tight_layout()
plt.show()

### Wpływ wyboru funkcji okna
Funkcja okna modyfikuje kształt ramki w celu zmniejszenia nieciągłości na jej końcach. W projekcie zaimplementowano 4 typy okien sprzętowych:
1. **Prostokątne (Rectangular):** Brak modyfikacji sygnału. Najwęższy płat główny (dobra rozdzielczość częstotliwościowa), ale bardzo wysokie płaty boczne (-13 dB), co prowadzi do silnego przecieku widma.
2. **Hann:** Klasyczny kompromis. Płaty boczne opadają szybko, poziom pierwszego płata bocznego to -32 dB.
3. **Blackman:** Bardzo niski poziom płatów bocznych (-58 dB), ale szerszy płat główny (nieco gorsza rozdzielczość). Doskonały do wykrywania słabych tonów obok silnych sygnałów.
4. **Flat Top:** Bardzo płaski płat główny, co pozwala na dokładny pomiar amplitudy prążków częstotliwościowych bez błędu wynikającego z pozycji prążka (scalloping loss). Bardzo wysokie tłumienie płatów bocznych, ale szeroki płat główny.

Poniższy kod demonstruje kształt tych czterech okien oraz ich charakterystyki widmowe w dziedzinie częstotliwości.

In [ ]:
# Wykres porównujący kształty okien zaimplementowane w projekcie
N = 512
w_rect = np.ones(N)
w_hann = np.sin(np.pi * np.arange(N) / (N - 1)) ** 2
w_blackman = 0.42 - 0.5 * np.cos(2 * np.pi * np.arange(N) / (N - 1)) + 0.08 * np.cos(4 * np.pi * np.arange(N) / (N - 1))
a0, a1, a2, a3, a4 = 0.21557895, 0.41663158, 0.277263158, 0.083578947, 0.006947368
w_flattop = (a0 - a1 * np.cos(2 * np.pi * np.arange(N) / (N - 1)) +
             a2 * np.cos(4 * np.pi * np.arange(N) / (N - 1)) -
             a3 * np.cos(6 * np.pi * np.arange(N) / (N - 1)) +
             a4 * np.cos(8 * np.pi * np.arange(N) / (N - 1)))

plt.figure(figsize=(10, 5))
plt.plot(w_rect, label='Prostokątne (Rectangular)', color='gray', linestyle='--')
plt.plot(w_hann, label='Hann', color='forestgreen')
plt.plot(w_blackman, label='Blackman', color='firebrick')
plt.plot(w_flattop, label='Flat Top', color='darkorange')
plt.title("Funkcje okien czasowych (N=512)")
plt.xlabel("Numer próbki")
plt.ylabel("Amplituda okna")
plt.legend()
plt.grid(True)
plt.show()

## 2. Wysokopoziomowa analiza systemu (Komunikacja Python-FPGA i architektura sprzętowa)

### Architektura komunikacyjna Python-FPGA
Akcelerator zaimplementowany został na platformie Kria KV260 przy użyciu frameworka **PYNQ**. Narzędzie to umożliwia wygodną integrację kodu Python (działającego w systemie Linux na rdzeniach ARM PS - Processing System) z logiką sprzętową PL (Programmable Logic) za pomocą biblioteki PYNQ API.

Główne kanały komunikacyjne przedstawiają się następująco:
1. **AXI Direct Memory Access (DMA):** Służy do szybkiego przesyłania bloków danych audio z pamięci DDR (kontrolowanej przez PS) bezpośrednio do akceleratora STFT (w PL) i odbierania wyników transformacji FFT z powrotem do DDR.
2. **AXI GPIO:** Służy do sterowania parametrami pracy w czasie rzeczywistym. Rejestr GPIO podłączony jest do portu wyboru typu okna (`window_type`) akceleratora STFT, co umożliwia przełączanie okien w locie bez przeładowywania bitstreamu.

### Przepływ danych w systemie (Schemat blokowy)

Poniższy diagram przedstawia drogę sygnału w systemie od wejścia audio na stacji roboczej po końcowy wykres:

```mermaid
graph TD
    PC[Stacja robocza PC: net_tx_wav / mic] -->|UDP: Float32 Audio| PS[ARM PS: Kria KV260]
    PS -->|Odbiór i Skalowanie: Int16 Q1.15| DDR[Bufor wejściowy: in_buffer DDR]
    DDR -->|AXI DMA MM2S| WRAP[Wrapper STFT: stft_axis_wrapper]
    WRAP -->|1. Windowing unit| WIN[windowing_unit.sv]
    WIN -->|2. Dual-port writes| PP[fft_ping_pong: 4x RAM Banks]
    PP <-->|3. Iterative Butterfly calculations| CORE[FFT Core: fft_fsm.sv + btf_radix2.sv]
    PP -->|4. Dual-port reads| OUT_MUX[Multiplexer wyjściowy]
    OUT_MUX -->|5. Packing: Im + Re| TX_FIFO[Rejestry AXI-Stream]
    TX_FIFO -->|AXI DMA S2MM| DDR_OUT[Bufor wyjściowy: out_buffer DDR]
    DDR_OUT -->|Rozpakowanie i Konwersja dB| PLOT[Plotly FigureWidget WebGL UI]
```

### Schemat połączeń Block Design w Vivado
Schemat połączeń Block Design w Vivado integruje rdzeń akceleratora `stft_axis_wrapper_0` z kontrolerem AXI DMA, blokiem GPIO oraz szyną AXI SmartConnect zapewniającą dostęp do pamięci DRAM:

![Schemat Block Design](../verilog_files/block_design.png)

### Format danych i protokół DMA
Komunikacja z AXI DMA odbywa się z użyciem fizycznie ciągłych buforów pamięci, alokowanych za pomocą metody `pynq.allocate`:
* **Bufor wejściowy (`in_buffer`):** Rozmiar 512 próbek, typ `np.int16` (16-bitowe liczby ze znakiem). Reprezentuje jedną ramkę czasową sygnału audio w formacie stałoprzecinkowym Q1.15.
* **Bufor wyjściowy (`out_buffer`):** Rozmiar 512 próbek, typ `np.int32`. Zawiera spakowane zespolone prążki FFT.
  * **Dolne 16 bitów (`[15:0]`):** Część rzeczywista (Real) jako 16-bitowa liczba ze znakiem (`int16`).
  * **Górne 16 bitów (`[31:16]`):** Część urojona (Imaginary) jako 16-bitowa liczba ze znakiem (`int16`).

Rozpakowanie danych w Pythonie realizowane jest wydajnie za pomocą widoku tablicy NumPy:
```python
real_part = (out_buffer & 0xFFFF).astype(np.uint16).view(np.int16)
imag_part = ((out_buffer >> 16) & 0xFFFF).astype(np.uint16).view(np.int16)
hw_fft = real_part + 1j * imag_part
```

---

### Koncepcyjna struktura i działanie kodu Verilog (PL)

Głównym wyzwaniem w projektowaniu akceleratora STFT/FFT o wysokiej przepustowości jest konieczność równoczesnego wykonywania trzech procesów:
1. **Odbierania** nowych próbek czasu z szyny DMA (MM2S).
2. **Obliczania** transformacji FFT na bieżącej ramce danych.
3. **Nadawania** obliczonych zespolonych wyników poprzedniej ramki z powrotem na szynę DMA (S2MM).

Gdybyśmy użyli pojedynczego bloku pamięci RAM, procesy te wchodziłyby w konflikty dostępowe (hazardy odczytu/zapisu), a rdzeń obliczeniowy musiałby być cyklicznie wstrzymywany (stall), co drastycznie obniżyłoby wydajność.

Aby rozwiązać ten problem, zaimplementowano zaawansowaną architekturę wielobankową sterowaną dedykowaną maszyną stanów.

---

### 1. Architektura Ping-Pong RAM (Pamięć wielobankowa)

W module `fft_ping_pong.sv` zadeklarowano **cztery fizyczne pamięci RAM Dual-Port** (`ram_0` do `ram_3`), każda o pojemności 512 słów 16-bitowych dla części rzeczywistej i urojonej. W każdym momencie czasu każda z pamięci pełni inną rolę (rolę tę definiują indeksy ról: `rx_idx`, `tx_idx`, `fft_a_idx`, `fft_b_idx`):

* **Bufor RX (`rx_idx`):** Pamięć, do której wpisywany jest strumień nowych próbek wejściowych przechodzących w locie przez jednostkę okienkowania.
* **Bufor TX (`tx_idx`):** Pamięć, z której odczytywane są gotowe wyniki FFT i przesyłane do DMA.
* **Bufor roboczy A (`fft_a_idx`) & Bufor roboczy B (`fft_b_idx`):** Pamięci robocze rdzenia FFT, między którymi następuje naprzemienny odczyt i zapis w kolejnych etapach (stages) motylków FFT.

Po zakończeniu pełnej transformacji (9 etapów) oraz całkowitym odebraniu nowej ramki wejściowej, następuje **rotacja ról buforów**:

```mermaid
graph LR
    subgraph KANAŁY ZEWNĘTRZNE
        IN[Wejście DMA RX]
        OUT[Wyjście DMA TX]
    end
    subgraph PAMIĘCI RAM BRAM
        R0[Bank 0: ram_0]
        R1[Bank 1: ram_1]
        R2[Bank 2: ram_2]
        R3[Bank 3: ram_3]
    end
    subgraph RDZEŃ FFT
        BTF[Motylek btf_radix2]
    end
    
    IN -->|Rotacja ról: rx_idx| R0
    OUT <--|Rotacja ról: tx_idx| R1
    BTF <-->|fft_a_idx| R2
    BTF <-->|fft_b_idx| R3
    
    style R0 fill:#f9f,stroke:#333,stroke-width:2px
    style R1 fill:#bbf,stroke:#333,stroke-width:2px
    style R2 fill:#bfb,stroke:#333,stroke-width:2px
    style R3 fill:#fbb,stroke:#333,stroke-width:2px
```

* **Odwracanie bitów w locie (Bit-Reversal):**
  W klasycznym algorytmie FFT Radix-2 DIT próbki wejściowe muszą zostać ułożone w kolejności bitowo-odwróconej. W naszym projekcie operacja ta jest realizowana sprzętowo **całkowicie bezkosztowo** podczas zapisu do bufora RX, poprzez zamianę kolejności linii adresowych:
  ```verilog
  assign write_addr_reversed[LOG2_N-1-k] = write_addr[k];
  ```
  Dzięki temu dane w pamięci roboczej są od razu poprawnie uporządkowane przed startem obliczeń FFT.

---

### 2. Maszyna stanów kontrolera FFT (`fft_fsm.sv`)

Zaimplementowany algorytm FFT dla $N=512$ punktów wymaga wykonania $log_2(512) = 9$ etapów (stages). W każdym etapie wykonujemy $N/2 = 256$ operacji motylkowych (butterflies). Za koordynację adresowania RAM i twiddle factors (współczynników obrotu) odpowiada maszyna stanów zdefiniowana w `fft_fsm.sv`, posiadająca 4 stany główne:

```mermaid
stateDiagram-v2
    [*] --> IDLE : Reset
    IDLE --> CALC : start = 1\nInicjalizacja: stage_cnt=1,\ngroup_cnt=0, bfly_cnt=0
    CALC --> CALC : Iteracja po bfly_cnt i group_cnt\ngenerowanie adresów A, B, W
    CALC --> FLUSH : Koniec etapu (stage)\nvalid = 0
    FLUSH --> CALC : drain_cnt >= 2\nstage_cnt < 9\nIncrement stage_cnt
    FLUSH --> DONE : drain_cnt >= 2\nstage_cnt == 9
    DONE --> IDLE : done = 1
```

* **IDLE:** Stan oczekiwania. Po otrzymaniu impulsu `start` (sygnalizującego zamianę banków i gotowość nowych danych), maszyna zeruje liczniki i przechodzi do stanu `CALC` dla pierwszego etapu (`stage_cnt = 1`).
* **CALC:** Stan generowania adresów. W każdym takcie wyliczane są dwa adresy odczytu z pamięci RAM (`addr_A` i `addr_B` odległe o szerokość kroku motylka w danym etapie) oraz adres twiddle factor (`addr_W`). Pętla wewnętrzna zwiększa `bfly_cnt`, a pętla zewnętrzna przesuwa `group_cnt`. Po zaadresowaniu wszystkich 256 motylków, stan przechodzi do `FLUSH`.
* **FLUSH:** Oczyszczanie potoku. Ze względu na latencję odczytu z BRAM (1 takt) oraz latencję rejestrów wewnątrz motylka obliczeniowego (1 takt), dane docierają do pamięci zapisywanej z 2-taktowym opóźnieniem. Maszyna stanów wstrzymuje adresowanie na 2 takty (`PIPELINE_DELAY`), dając czas ostatnim próbkom na zapis. Następnie, jeśli to nie był 9. etap, następuje inkremetacja `stage_cnt` i powrót do `CALC`.
* **DONE:** Sygnalizuje ukończenie pełnej transformacji impulsem `done` i powraca do stanu `IDLE`.

---

### 3. Jednostka motylkowa FFT Radix-2 (`btf_radix2.sv`)

Operacja motylkowa jest podstawowym elementem obliczeniowym FFT. Przyjmuje ona dwie liczby zespolone $A$ i $B$, współczynnik obrotu (twiddle factor) $W = e^{-j\theta}$ i wylicza:
$$out_A = A + B \cdot W$$
$$out_B = A - B \cdot W$$

W celu uzyskania maksymalnej częstotliwości taktowania, moduł `btf_radix2.sv` jest w pełni potokowy (pipelined):
* **Takt 1 (Mnożenie):** Realizowane jest pełne mnożenie zespolone $B \times W$:
  $$Re(B \cdot W) = B_{re} \cdot W_{re} - B_{im} \cdot W_{im}$$
  $$Im(B \cdot W) = B_{re} \cdot W_{im} + B_{im} \cdot W_{re}$$
  W tym samym takcie sygnał z kanału $A$ jest opóźniany (`A_re_delay`), aby zsynchronizować go z wyjściem z mnożników.
* **Takt 2 (Sumowanie i skalowanie):** Wyniki mnożenia są sumowane/odejmowane kombinacyjnie od opóźnionego sygnału $A$. Ponieważ współczynniki $W$ zapisane są w formacie Q1.15, wyniki mnożenia są skalowane z formatu Q2.30 z powrotem do Q1.15 poprzez wycięcie bitów `[30:15]`:
  ```verilog
  assign out_A_re = A_re_delay + mult_B_W_re[30:15];
  assign out_B_re = A_re_delay - mult_B_W_re[30:15];
  ```


## 3. Działanie systemu w czasie rzeczywistym (Real-Time Spectrograph)

### Architektura wielowątkowa aplikacji odbiorczej
Wizualizacja widma w czasie rzeczywistym wymaga bezlatencyjnej pracy. W tym celu zaimplementowano architekturę wielowątkową w Pythonie opartą na podziale **Producer-Consumer** (Producent-Konsument) ze współdzielonym buforem kołowym:

1. **Wątek tła (Producer):**
   * Nasłuchuje na porcie UDP `5005` na pakiety danych wysyłane ze stacji roboczej PC (np. z mikrofonu przy użyciu `net_tx_mic.py` lub z pliku audio za pomocą `net_tx_wav.py`).
   * Pakiety UDP zawierają ramkę audio (2048 próbek typu float32). Wątek pobiera z nich środkowy fragment o długości 512 próbek.
   * Skaluje sygnał do zakresu `int16`, ładuje go do bufora `in_buffer` i uruchamia sprzętową akcelerację DMA na FPGA.
   * Po zakończeniu transferu odbiera spakowane widmo zespolone, oblicza jego moduł i konwertuje na decybele (dB) z uwzględnieniem wzmocnienia wybranego okna (normalizacja amplitudy).
   * Zapisuje nową kolumnę danych do bufora kołowego `waterfall` reprezentującego historię czasową spektrogramu i aktualizuje wskaźnik zapisu.
2. **Wątek główny wizualizacji (Consumer):**
   * Działa z częstotliwością ok. 12 FPS (odstęp rysowania `DRAW_INTERVAL = 0.08 s`).
   * Pobiera aktualną historię spektrogramu z bufora kołowego, odpowiednio porządkuje kolumny czasowe i przekazuje je do widgetu `Plotly FigureWidget` opartego na technologii WebGL.
   * Użycie WebGL pozwala na płynną animację gęstej siatki punktów ($257 \times 100$) bez obciążania procesora.

### Kluczowa optymalizacja WebSocket
Początkowo przesyłanie surowych danych typu `float32` przez protokół WebSocket do widgetu Jupyter powodowało narzut komunikacyjny, prowadzący do opóźnień (jedna sekunda rzeczywista trwała na wykresie ponad dwie sekundy) i w konsekwencji do przepełnienia bufora UDP socketu.
Zastosowano **krytyczną optymalizację**: dane spektrogramu są rzutowane na typ `int16` bezpośrednio przed wysłaniem do widgetu Plotly:
```python
ordered_waterfall = np.concatenate((waterfall[:, curr_ptr:], waterfall[:, :curr_ptr]), axis=1).astype(np.int16)
```
Casting ten eliminuje nadmiarowe znaki po przecinku w formacie tekstowym JSON przesyłanym przez serwer Jupyter (skrócenie payloadu o **około 70%**), co pozwoliło na osiągnięcie pełnej płynności i całkowite zlikwidowanie opóźnień.

### Interaktywność UI i dynamiczna zmiana okien
W interfejsie umieszczono widget rozwijany (`widgets.Dropdown`), który pozwala na dynamiczną zmianę okna w locie. Wybrana opcja wyzwala zapis odpowiedniej wartości do rejestru GPIO na FPGA (0: Prostokątne, 1: Hann, 2: Blackman, 3: Flat Top).
Aplikacja automatycznie dopasowuje współczynnik normalizacji amplitudy w kodzie Python, zapobiegając pozornej zmianie poziomu głośności sygnału na spektrogramie przy przełączaniu okien.

### Weryfikacja działania sprzętowego
Wyniki uzyskane z akceleratora FPGA zostały zweryfikowane poprzez bezpośrednie porównanie z programowym modelem referencyjnym w Pythonie (NumPy). Poniższy wykres (znajdujący się również w folderze projektu) przedstawia widmo sygnału testowego dla wszystkich 4 typów okien obliczone równolegle w sprzęcie oraz oprogramowaniu:

![Weryfikacja sprzętowa](../files_on_fpga_jupyter/stft_hardware_verification.png)

Jak widać, widma pokrywają się niemal idealnie. Błąd średniokwadratowy (MSE) wynosi:
* **Rectangular:** 1012.96
* **Hann:** 811.57
* **Blackman:** 775.71
* **Flat Top:** 774.04
Niewielkie różnice wynikają z kwantyzacji stałoprzecinkowej Q1.15 stosowanej w strukturach FPGA.

### Prezentacja działania w czasie rzeczywistym
Pełna wideo-prezentacja działania systemu akceleracji STFT sprzężonego ze skryptem wejściowym mikrofonu znajduje się pod poniższym adresem:

🎥 **[Prezentacja wideo na YouTube](https://www.youtube.com/watch?v=Gt5P3kQTfR4)**

---

## 4. Podsumowanie i Wnioski
Zrealizowany projekt udowadnia wysoką użyteczność sprzętowo-programowej akceleracji algorytmów DSP na nowoczesnych platformach SoC (takich jak Kria KV260). Dzięki podziałowi zadań:
* **FPGA (PL):** realizuje krytyczne czasowo, powtarzalne obliczenia stałoprzecinkowe (okienkowanie, buforowanie ping-pong, FFT) z gwarantowanym czasem wykonania i minimalnym poborem mocy.
* **Procesor ARM (PS - Python):** zarządza odbiorem strumienia UDP z sieci, przetwarza wyjście widma na skalę logarytmiczną oraz obsługuje bogaty graficznie interfejs użytkownika w przeglądarce.

Błąd średniokwadratowy (MSE) między wynikami z akceleratora a referencyjną implementacją zmiennoprzecinkową w NumPy wyniósł poniżej 1000 dla wszystkich typów okien, co przy skali wartości energii widma potwierdza wysoką precyzję obliczeń stałoprzecinkowych Q1.15. Wprowadzona optymalizacja serializacji danych WebSocket umożliwiła stabilne działanie spektrogramu w czasie rzeczywistym z częstotliwością odświeżania 12 FPS przy próbkowaniu sygnału audio 44.1 kHz.